# Supervisor Multi-AI-Agent System with LangGraph

In `01_Simple_Multi_Agent` the agents ran in a **fixed order**: researcher, then writer, every time. That works when every question needs the same steps. Real tasks vary: some need more research, some need analysis before writing, and some can be answered straight away.

A **supervisor** system adds a coordinating agent that **decides at run time** which specialist works next. Each specialist does its part, control comes back, and the supervisor keeps routing until the job is done.

![Supervisor multi-AI-agent architecture](images/supervisor_multi_agent.png)

## Reading the diagram

- **`__start__ → Supervisor`**: every request starts at the supervisor.
- **Solid arrows (route)**: the supervisor sets `next_agent`, and the run moves to that agent.
- **Dashed arrows (report back)**: the agent adds its reply to `messages`, and control returns to the supervisor.
- **`next_agent = "end"`**: once the work is done, the supervisor routes to `__end__`, and the last reply is the final answer.
- **Shared state**: every agent reads and writes the same state, so each one sees the work done before it.

## The agents

| Agent | Job |
|---|---|
| **Supervisor** | Reads the conversation so far and picks who works next: `researcher`, `analyst`, `writer`, or `end` |
| **Researcher** | Gathers the facts and sources for the question |
| **Analyst** | Examines the research: key points, comparisons, gaps, and conclusions |
| **Writer** | Turns the research and analysis into the final answer for the user |

## Simple pipeline vs supervisor

| | Simple pipeline (`01`) | Supervisor (`02`) |
|---|---|---|
| Order of agents | Fixed by `add_edge` | Chosen at run time by the supervisor |
| Skip or repeat an agent | No | Yes |
| LLM calls per run | One per agent | One per agent **plus** one per supervisor decision |
| Easy to predict | Very | Less, so check the trace in LangSmith |

## Core building blocks

| Concept | What it is | In LangGraph |
|---|---|---|
| **Routing field** | Where the supervisor stores its decision | `next_agent: str` in the state |
| **Supervisor node** | An LLM call that returns the next agent's name | Limit the answer to `Literal["researcher", "analyst", "writer", "end"]` so it can only pick a valid name |
| **Routing** | Sends the run to the chosen agent | `add_conditional_edges("supervisor", route_fn)` where `route_fn` reads `next_agent`, or a node that returns `Command(goto="agent_name")` |
| **Hand back** | A specialist gives control back after its turn | An edge (or `Command`) from each specialist back to `supervisor` |
| **Stop condition** | How the run ends | The supervisor picks `"end"`, which routes to `END` |

## Steps

1. **Define the state**: `messages` plus `next_agent`.
2. **Create the LLM** and write a short system prompt for each specialist.
3. **Write the supervisor node**: give it the conversation and the list of agents, and have it return `next_agent`.
4. **Write the specialist nodes** (researcher, analyst, writer): each adds its reply under `messages` and hands control back.
5. **Add the routing** with conditional edges or `Command(goto=...)`.
6. **Compile, draw the graph, run it**, and trace it in LangSmith to see the order the supervisor chose.

## Things to watch

- **Give the supervisor a clear way to finish**, for example "choose `end` once the writer has produced the final answer". Without one, it may keep calling agents.
- **Cap the number of steps** so a confused supervisor cannot loop for ever: `graph.invoke(inputs, config={"recursion_limit": 15})`.
- **Restrict the choices to a `Literal`**, so a reply like `"Researcher"` or `"the analyst"` cannot break the routing.
- **Label each agent's reply** with `name="researcher"` (and so on), so the supervisor can see who has already done what.

In [14]:
import os
from typing import TypedDict, Annotated, List, Dict, Any, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent, ToolNode, tools_condition
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import ChatPromptTemplate
import random
from datetime import datetime
from dotenv import load_dotenv

print("Import Successful")

Import Successful


In [15]:
## Define the  State

class SupervisorState(MessagesState):
    """State for the supervisor agent, including messages and context."""
    next_agent: str ="" # which agent is currently active
    research_data:str = "" # date of the research
    analysis: str = "" # analysis of the research
    final_report: str = "" # final report generated by the supervisor
    task_complete: bool = False # flag to indicate if the task is complete
    current_task: str = "" # current task being performed by the agent

In [16]:
## define llm 
def llm(model: str = "gpt-5", temperature: float = 0.7) -> ChatOpenAI:
    """Create an OpenAI model with the specified parameters."""
    return ChatOpenAI(model=model, temperature=temperature)

In [17]:
## create supervisor with openai llm
def create_supervisor_chain():
    """Create the supervisor decision chain with OpenAI LLM."""

    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system","""You are a supervior managing a team of AI Agents:
        
        1. Research Agent: Responsible for gathering information and data on the assigned topic.
        2. Analyst Agent: Responsible for analyzing the gathered information and providing insights.
        3. Writer Agent: Responsible for compiling the analysis into a comprehensive report.
        
        Based on teh current state and conversations, decide which agent should be called next and provide the necessary instructions for that agent.
        If the task is complee, responsd with 'DONE' and provide the final report.
        
        Current state:
        - Has research data : {has_research}
        - Has analysis : {has_analysis}
        - Has final report : {has_final_report}
        
        Response with ONLY the agent name (researcher/analyst/writer) or DONE"""),
        ("human","{task}")

    ])

    return supervisor_prompt | llm()

In [18]:
## Define supervisor agent
def create_supervisor_agent(state:SupervisorState) -> Dict:
    """Supervisor decides next agent usign openai LLM"""

    messages = state["messages"]
    task = messages[-1].content if messages else "No task"

    ## check what has been completed

    has_research = bool(state.get("research_data",""))
    has_analysis = bool(state.get("analysis",""))
    has_final_report = bool(state.get("final_report"))

    # Get LLM Decision
    chain = create_supervisor_chain()
    decision = chain.invoke({
        "task":task,
        "has_research": has_research,
        "has_analysis": has_analysis,
        "has_final_report": has_final_report
    })

    ## Parse decision
    decision_text = decision.content.strip().lower()
    print(decision_text)